In [ ]:
# %% [markdown]
# # DeepFM 模型验证
# ## 目的：在小数据集上验证模型结构正确性

# %%
import sys
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# 设置随机种子确保可复现
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================
# 1. 添加项目路径
# ============================================================
project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"✅ 已添加项目路径: {project_root}")

# ============================================================
# 2. 导入模块
# ============================================================
from deepfm_ctr.DataProcess import (
    DataProcessor, CATEGORICAL_FEATURES, NUMERIC_FEATURES, LABEL
)
from deepfm_ctr.config import (
    DATA_CONFIG,
    MODEL_CONFIG,
    TRAINING_CONFIG
)
from deepfm_ctr.model import DeepFMBuilder
from deepfm_ctr.trainer import ModelTrainer

print("✅ 导入成功")

# %%
# ============================================================
# 3. 加载小样本数据（快速验证）
# ============================================================
print("="*60)
print("加载数据")
print("="*60)

# 使用 DATA_CONFIG 初始化 DataProcessor
processor = DataProcessor(
    data_dir=DATA_CONFIG['data_dir'],
    start_date=DATA_CONFIG['start_date'],
    split_date=DATA_CONFIG['split_date'],
    end_date=DATA_CONFIG['end_date'],
    val_ratio=DATA_CONFIG['val_ratio'],
    seed=DATA_CONFIG['seed']
)

# 加载并预处理数据（只用1%数据验证）
train_df, val_df, test_df = processor.load_and_preprocess(
    sample_rate=0.01  # 只用1%数据验证
)

print(f"✅ 小样本验证:")
print(f"   训练集: {len(train_df):,}")
print(f"   验证集: {len(val_df):,}")
print(f"   测试集: {len(test_df):,}")

# ============================================================
# 强制检查并修复所有类别特征中的非法值（新增）
# ============================================================
print("\n" + "="*60)
print("🔧 强制检查并修复类别特征")
print("="*60)

# 类别特征由 DataProcess.py 唯一定义

# 对 train, val, test 分别检查和修复
for df_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"\n📊 检查 {df_name} 集:")
    for col in CATEGORICAL_FEATURES:
        if col not in df.columns:
            continue
        
        # 检查是否有非法值
        invalid_mask = (df[col] == -2147483648) | (df[col] == -999999999) | (df[col] == 999999999)
        invalid_count = invalid_mask.sum()
        
        if invalid_count > 0:
            print(f"   ⚠️ {col}: 发现 {invalid_count} 个非法值")
            # 替换为 0
            df.loc[invalid_mask, col] = 0
            print(f"   ✅ {col}: 已修复")

# 再次检查
print("\n" + "="*60)
print("🔍 验证修复结果")
print("="*60)

for df_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"\n{df_name} 集:")
    for col in CATEGORICAL_FEATURES:
        if col in df.columns:
            invalid_count = ((df[col] == -2147483648) | (df[col] == -999999999) | (df[col] == 999999999)).sum()
            if invalid_count > 0:
                print(f"   ❌ {col}: 仍有 {invalid_count} 个非法值！")
            else:
                print(f"   ✅ {col}: 无非法值")

print("\n✅ 修复完成！")
# %%
# ============================================================
# 4. 获取类别特征的词汇表大小
# ============================================================
print("\n" + "="*60)
print("构建模型")
print("="*60)

# 词汇表大小由 DataProcessor 唯一生成
vocabularies = processor.vocabularies

print(f"词汇表大小: {processor.vocab_sizes}")

# %%
# ============================================================
# 5. 构建模型
# ============================================================
builder = DeepFMBuilder(
    categorical_features=CATEGORICAL_FEATURES,
    numeric_features=NUMERIC_FEATURES,
    embed_dim=MODEL_CONFIG['embed_dim'],
    dnn_hidden_units=MODEL_CONFIG['dnn_hidden_units'],
    dropout_rate=MODEL_CONFIG['dropout_rate'],
    learning_rate=MODEL_CONFIG['learning_rate'],
    seed=DATA_CONFIG['seed']
)

model = builder.build_model(vocabularies)
model.summary()  # 检查模型结构

# %%
# ============================================================
# 6. 训练模型（快速验证）
# ============================================================
print("\n" + "="*60)
print("训练模型")
print("="*60)

trainer = ModelTrainer(
    model=model,
    batch_size=TRAINING_CONFIG['batch_size'],
    epochs=5,  # 只训练5个epoch验证
    patience=TRAINING_CONFIG['early_stopping_patience']
)

history = trainer.train(
    processor.make_model_inputs(train_df), processor.make_labels(train_df),
    processor.make_model_inputs(val_df), processor.make_labels(val_df),
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', 
        mode='max', 
        patience=2, 
        restore_best_weights=True
    )]
)

print("✅ 训练完成")

# %%
# ============================================================
# 7. 可视化训练曲线
# ============================================================
print("\n" + "="*60)
print("训练曲线")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss曲线
axes[0].plot(history.history['loss'], label='train_loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='val_loss', linewidth=2)
axes[0].set_title('Loss曲线')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUC曲线
axes[1].plot(history.history['auc'], label='train_auc', linewidth=2)
axes[1].plot(history.history['val_auc'], label='val_auc', linewidth=2)
axes[1].set_title('AUC曲线')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# %%
# ============================================================
# 8. 评估模型
# ============================================================
print("\n" + "="*60)
print("模型评估")
print("="*60)

# 评估测试集
metrics = trainer.evaluate(
    processor.make_model_inputs(test_df),
    processor.make_labels(test_df),
    processor.make_groups(test_df),
)
print(f"测试集 AUC: {metrics['auc']:.4f}")
print(f"测试集 GAUC: {metrics['gauc']:.4f}")

# %%
# ============================================================
# 9. 验证预测形状
# ============================================================
print("\n" + "="*60)
print("预测验证")
print("="*60)

# 准备测试数据
x_test = processor.make_model_inputs(test_df)
y_test = processor.make_labels(test_df)
test_predictions = model.predict(x_test, batch_size=1024, verbose=0)

print(f"预测形状: {test_predictions.shape}")
print(f"预测范围: [{test_predictions.min():.4f}, {test_predictions.max():.4f}]")
print(f"预测均值: {test_predictions.mean():.4f}")
print(f"预测标准差: {test_predictions.std():.4f}")

# %%
# ============================================================
# 10. 单样本预测调试
# ============================================================
print("\n" + "="*60)
print("单样本预测")
print("="*60)

# 取第一个样本
single_sample = test_df.iloc[[0]]
x_single = processor.make_model_inputs(single_sample)
pred = model.predict(x_single, verbose=0)

print(f"单样本预测值: {pred[0][0]:.6f}")
print(f"真实标签: {single_sample[LABEL].values[0]}")

# %%
print("\n✅ 模型验证完成！")